### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [2]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [3]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [4]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [6]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [7]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="google/gemma-4-e4b",
                                            base_url="http://10.54.22.157:1234/v1",
                                            api_key="lmstudio",
                                            model_info={
                                                    "family": "unknown",
                                                    # "supports_chat_completions": True,
                                                    "vision": True,
                                                    "function_calling": True,
                                                    "json_output": True,
                                                    "structured_output": True,
                                            })
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="google/gemma-4-e4b",
                                            base_url="http://10.54.22.157:1234/v1",
                                            api_key="lmstudio",
                                            model_info={
                                                    "family": "unknown",
                                                    # "supports_chat_completions": True,
                                                    "vision": True,
                                                    "function_calling": True,
                                                    "json_output": True,
                                                    "structured_output": True,
                                            })
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="google/gemma-4-e4b",
                                            base_url="http://10.54.22.157:1234/v1",
                                            api_key="lmstudio",
                                            model_info={
                                                    "family": "unknown",
                                                    # "supports_chat_completions": True,
                                                    "vision": True,
                                                    "function_calling": True,
                                                    "json_output": True,
                                                    "structured_output": True,
                                            })
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [8]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [9]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [10]:
display(Markdown(response.content))

## Pros of AutoGen:
Based on research, here are the primary reasons (pros) for considering AutoGen for your new AI Agent project:

### Advantages of Using AutoGen

1.  **Multi-Agent Orchestration:**
    *   AutoGen is designed from the ground up to manage *interactions* between multiple autonomous agents. This makes it ideal for complex tasks that require collaboration, where different specialized "agents" (e.g., a code generator agent, a reviewer agent, and a planner agent) work together toward a common goal.

2.  **Scalability and Modularity:**
    *   The framework is highly modular and extensible. You can easily add new types of agents or change the communication protocol without rewriting the core system. This allows your project to scale as its complexity increases.

3.  **Flexibility in Interaction Patterns (Conversational Agents):**
    *   It facilitates various interaction patterns—from simple request/response loops to complex, multi-turn conversations with defined roles and conversational flows. You can define sophisticated workflows where agents critique each other's outputs or debate solutions until consensus is reached.

4.  **Ease of Use and Observability:**
    *   AutoGen provides integrated tooling for monitoring agent behavior. This means that debugging a complex workflow—figuring out *why* the conversation stalled or *which* agent failed—is simplified through built-in observability tools, greatly reducing development overhead.

## Cons of AutoGen:
Based on research regarding potential drawbacks or cons of using AutoGen for an AI Agent project, here is a brief summary:

*   **Documentation and Usability:** A common critique is that the documentation can be difficult to follow, sometimes lacking in detailed examples or practical use cases, which can create a steep learning curve.
*   **Functional Limitations:** Some users report issues with certain functionalities, such as handling structured outputs reliably within the agent workflows.
*   **Competition and Differentiation:** The ecosystem of AI agents is rapidly evolving (e.g., tools like `ag2`), and AutoGen faces scrutiny regarding how it differentiates its core capabilities from other similar frameworks or applications.

In essence, potential challenges revolve around setup complexity, occasional functional bugs or limitations, and keeping pace with a very competitive and quickly evolving market landscape.



## Decision:

**Decision:** Use AutoGen.

**Rationale:**
Based purely on the research, AutoGen's core strengths—specifically its native **Multi-Agent Orchestration**, **Scalability**, and support for **Flexible Interaction Patterns**—make it exceptionally well-suited for a complex AI Agent project requiring collaboration (e.g., critique loops or debate). While we must account for the noted steep learning curve due to documentation gaps, the inherent value provided by its specialized design for managing interactions between multiple autonomous agents is too significant to ignore. The benefits of advanced orchestration and integrated observability outweigh the initial development overhead associated with mastering the framework's complexity.

In [11]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [12]:
await host.stop()